# HX2605 data audit
Run from the project root or notebooks directory. No network calls and no API credentials.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.common import load_config
from src.db.loaders import connect
from src.dataset.snapshots import frame
import pandas as pd
cfg = load_config("configs/live_pilot.yaml")
con = connect(cfg["database"])

In [ ]:
counts = {t: con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ["markets","prices","documents","snapshots","resolutions","forecasts"]}
pd.Series(counts, name="records")

In [ ]:
data = frame(con)
data.head()

In [ ]:
pd.read_sql_query("SELECT source,source_type,content_kind,COUNT(*) AS records FROM documents GROUP BY source,source_type,content_kind",con)

In [ ]:
pd.read_sql_query("SELECT stage,status,detail_json FROM pipeline_events ORDER BY event_id DESC LIMIT 20",con)

Check exclusions and coverage before training. Article count is not the number of independent forecasting events. Historical records fetched today may be inadmissible at old cutoffs.

In [ ]:
con.close()